In [3]:
import pandas as pd

"""
Rule-Based College Library Assistant Chatbot
Features: Book search, fine calculation, account view, book renewal
Includes: Mathematical calculations, if-else conditions, exception handling
"""

# ─── Data ────────────────────────────────────────────────────────────────────

FINE_PER_DAY = 5          # ₹5 per day
MAX_BOOKS    = 5          # Max books a student can borrow
LOAN_DAYS    = 14         # Loan period in days

STUDENT_DATA = {
    "STU001": {
        "name":     "Mahima Chavan",
        "borrowed": ["Introduction to Algorithms", "Data Structures in C++"],
        "due_in":   [-3, 5],   # negative = overdue by N days
    },
    "STU002": {
        "name":     "Paul Kale",
        "borrowed": ["Calculus Volume 1"],
        "due_in":   [2],
    },
    "STU003": {
        "name":     "Stuti Ghadge",
        "borrowed": [],
        "due_in":   [],
    },
}

BOOK_CATALOG = {
    "algorithms":      {"title": "Introduction to Algorithms",  "author": "Cormen et al.",     "copies": 2, "available": 1, "shelf": "CS-A3"},
    "calculus":        {"title": "Calculus Volume 1",           "author": "Tom Apostol",       "copies": 3, "available": 2, "shelf": "MTH-B2"},
    "data structures": {"title": "Data Structures in C++",      "author": "Mark Weiss",        "copies": 2, "available": 0, "shelf": "CS-A5"},
    "physics":         {"title": "University Physics",          "author": "Young & Freedman",  "copies": 4, "available": 3, "shelf": "PHY-C1"},
    "chemistry":       {"title": "Organic Chemistry",           "author": "Paula Bruice",      "copies": 2, "available": 2, "shelf": "CHM-D2"},
    "networking":      {"title": "Computer Networks",           "author": "Tanenbaum",         "copies": 3, "available": 1, "shelf": "CS-A7"},
}

LIBRARY_HOURS = {
    "Monday – Friday": "8:00 AM – 10:00 PM",
    "Saturday":        "9:00 AM – 8:00 PM",
    "Sunday":          "10:00 AM – 6:00 PM",
    "Exam Days":       "Reading hall open until 11:00 PM",
}

# ─── Helpers ─────────────────────────────────────────────────────────────────

def separator(char="─", width=55):
    print(char * width)

def header(text):
    separator()
    print(f"  {text}")
    separator()

def validate_student_id(sid: str) -> dict:
    """
    Validates format and existence of a student ID.
    Raises ValueError with a descriptive message on failure.
    """
    if not sid.startswith("STU") or not sid[3:].isdigit() or len(sid) != 6:
        raise ValueError(f"Invalid ID format '{sid}'. Expected format: STU001")
    if sid not in STUDENT_DATA:
        raise KeyError(f"Student ID '{sid}' not found in the system.")
    return STUDENT_DATA[sid]

# ─── Feature Functions ────────────────────────────────────────────────────────

def calculate_fine():
    """
    Mathematical Calculation:
    Fine = overdue_days × FINE_PER_DAY  (per book)
    Total Fine = sum of all per-book fines
    """
    header("📋  Overdue Fine Calculator")
    sid = input("  Enter your Student ID (e.g. STU001): ").strip().upper()

    try:
        student = validate_student_id(sid)
    except (ValueError, KeyError) as e:
        print(f"\n  ❌ Error: {e}\n")
        return

    name     = student["name"]
    borrowed = student["borrowed"]
    due_in   = student["due_in"]

    overdue_books = [(book, abs(days)) for book, days in zip(borrowed, due_in) if days < 0]

    if not overdue_books:
        print(f"\n  ✅ {name}, you have NO overdue books. All returns are on time!\n")
        return

    print(f"\n  Fine Summary for {name}:\n")
    total_fine = 0

    for book, days in overdue_books:
        # Mathematical calculation: fine per book
        fine = days * FINE_PER_DAY
        total_fine += fine
        print(f"  📕 {book}")
        print(f"     Overdue by {days} day(s)  ×  ₹{FINE_PER_DAY}/day  =  ₹{fine}")

    total_days = sum(d for _, d in overdue_books)
    print(f"\n  {'─'*40}")
    print(f"  Total Fine = ₹{FINE_PER_DAY} × {total_days} days = ₹{total_fine}")
    print(f"  {'─'*40}")
    print("  ⚠  Please pay at the library counter.")
    print("  ⚠  Fine doubles after 30 days overdue.\n")


def search_book():
    """
    If-Else Conditions:
    Checks partial keyword match against catalog keys and titles.
    Displays availability status based on copies available.
    """
    header("🔍  Book Search")
    query = input("  Enter book title or keyword: ").strip().lower()

    try:
        if len(query) < 2:
            raise ValueError("Search query is too short. Please enter at least 2 characters.")

        matches = [
            book for key, book in BOOK_CATALOG.items()
            if query in key or query in book["title"].lower()
        ]

        if not matches:
            print(f"\n  😕 No results found for '{query}'.")
            print("  Try keywords like: algorithms, calculus, physics, chemistry, networking\n")
            return

        print(f"\n  Found {len(matches)} result(s):\n")
        for book in matches:
            # If-else: determine availability label
            if book["available"] > 0:
                status = f"✅ Available  ({book['available']} of {book['copies']} copies)"
            else:
                status = "❌ All copies currently checked out"

            print(f"  📗 {book['title']}")
            print(f"     Author : {book['author']}")
            print(f"     Shelf  : {book['shelf']}")
            print(f"     Status : {status}\n")

    except ValueError as e:
        print(f"\n  ⚠  {e}\n")


def view_account():
    """
    If-Else Conditions:
    Determines due status of each borrowed book and calculates outstanding fine.
    """
    header("🎓  My Library Account")
    sid = input("  Enter your Student ID (e.g. STU001): ").strip().upper()

    try:
        student = validate_student_id(sid)
    except (ValueError, KeyError) as e:
        print(f"\n  ❌ Error: {e}\n")
        return

    name     = student["name"]
    borrowed = student["borrowed"]
    due_in   = student["due_in"]

    print(f"\n  Account: {name}")
    print(f"  Books borrowed: {len(borrowed)} / {MAX_BOOKS}\n")

    if not borrowed:
        print("  You have no books currently borrowed.\n")
        return

    total_fine = 0

    for book, days in zip(borrowed, due_in):
        # If-else chain for due status
        if days < 0:
            fine = abs(days) * FINE_PER_DAY
            total_fine += fine
            status = f"⚠  OVERDUE by {abs(days)} day(s)  |  Fine: ₹{fine}"
        elif days == 0:
            status = "🔔 Due TODAY — please return immediately"
        elif days <= 3:
            status = f"⏳ Due in {days} day(s)  — return soon"
        else:
            status = f"✅ Due in {days} day(s)"

        print(f"  📕 {book}")
        print(f"     {status}\n")

    if total_fine > 0:
        print(f"  {'─'*40}")
        print(f"  💰 Total Outstanding Fine: ₹{total_fine}")
        print(f"  {'─'*40}\n")


def renew_book():
    """
    If-Else Conditions + Exception Handling:
    Blocks renewal if overdue books exist; otherwise extends due dates.
    """
    header("🔄  Book Renewal")
    sid = input("  Enter your Student ID (e.g. STU001): ").strip().upper()

    try:
        student = validate_student_id(sid)
    except (ValueError, KeyError) as e:
        print(f"\n  ❌ Error: {e}\n")
        return

    name     = student["name"]
    borrowed = student["borrowed"]
    # We no longer need to assign 'due_in' to a local variable if we update in-place
    # due_in   = student["due_in"]

    if not borrowed:
        print(f"\n  ℹ  {name}, you have no books to renew.\n")
        return

    # If any book is overdue, block renewal
    if any(days < 0 for days in student["due_in"]): # Use student["due_in"] directly
        print(f"\n  ⛔ {name}, renewal is BLOCKED.")
        print("  You have overdue books. Please clear all fines at the counter first.\n")
        return

    print(f"\n  ✅ Renewal successful for {name}!\n")
    # Iterate with index to update the due_in list in place
    for i in range(len(borrowed)):
        book = borrowed[i]
        old_due_days = student["due_in"][i]
        new_due_days = old_due_days + LOAN_DAYS
        student["due_in"][i] = new_due_days # Update the specific element in the list

        print(f"  📕 {book}")
        print(f"     Extended by {LOAN_DAYS} days  →  new due: {new_due_days} day(s) from today\n")

    print("  ℹ  Note: Only one renewal is allowed per book per semester.\n")


def show_library_hours():
    header("🕐  Library Hours")
    for day, timing in LIBRARY_HOURS.items():
        print(f"  {day:<20} {timing}")
    print()


def show_rules():
    header("📌  Library Rules")
    print(f"  • Max books per student : {MAX_BOOKS}")
    print(f"  • Loan period           : {LOAN_DAYS} days")
    print(f"  • Overdue fine          : ₹{FINE_PER_DAY} per book per day")
    print(f"  • Renewals allowed      : 1 per book per semester")
    print(f"  • Renewal blocked if any overdue book exists\n")


def show_help():
    header("📖  Help — Available Commands")
    commands = [
        ("1 / fine",    "Calculate overdue fine"),
        ("2 / search",  "Search for a book"),
        ("3 / account", "View your borrowing account"),
        ("4 / renew",   "Renew borrowed books"),
        ("5 / hours",   "Library opening hours"),
        ("6 / rules",   "Library rules & limits"),
        ("help",        "Show this help menu"),
        ("exit / quit", "Exit the chatbot"),
    ]
    for cmd, desc in commands:
        print(f"  {cmd:<20} {desc}")
    print()

# ─── Main Loop ────────────────────────────────────────────────────────────────

def greet():
    separator("═")
    print("  🏛   COLLEGE LIBRARY ASSISTANT CHATBOT")
    separator("═")
    print("  Welcome! I can help you with:")
    print("  Book searches, fine calculations, your account, and more.")
    print("  Type  help  to see all commands.\n")

def route(user_input: str) -> bool:
    """
    Routes user input to the correct feature using if-else conditions.
    Returns False when user wants to exit, True otherwise.
    """
    cmd = user_input.strip().lower()

    if cmd in ("exit", "quit", "bye", "q"):
        print("\n  👋 Goodbye! Happy studying! 📚\n")
        separator("═")
        return False

    elif cmd in ("1", "fine", "overdue", "penalty", "calculate fine"):
        calculate_fine()

    elif cmd in ("2", "search", "find", "book", "available"):
        search_book()

    elif cmd in ("3", "account", "my books", "my account", "borrowed"):
        view_account()

    elif cmd in ("4", "renew", "extend"):
        renew_book()

    elif cmd in ("5", "hours", "timing", "open", "schedule"):
        show_library_hours()

    elif cmd in ("6", "rules", "limit", "policy"):
        show_rules()

    elif cmd in ("help", "?", "menu", "commands"):
        show_help()

    elif cmd in ("hi", "hello", "hey"):
        print("\n  👋 Hello! Type  help  to see what I can do.\n")

    elif cmd == "":
        pass  # ignore empty input

    else:
        print(f"\n  🤔 I didn't understand '{user_input}'.")
        print("  Type  help  to see available commands.\n")

    return True


def main():
    greet()
    while True:
        try:
            user_input = input("  You ▶  ").strip()
            if not route(user_input):
                break
        except KeyboardInterrupt:
            print("\n\n  Interrupted. Goodbye! 👋\n")
            break
        except Exception as e:
            # Top-level exception handler — catches any unexpected errors
            print(f"\n  ⚠  An unexpected error occurred: {e}")
            print("  Please try again or type  help  for commands.\n")


if __name__ == "__main__":
    main()

═══════════════════════════════════════════════════════
  🏛   COLLEGE LIBRARY ASSISTANT CHATBOT
═══════════════════════════════════════════════════════
  Welcome! I can help you with:
  Book searches, fine calculations, your account, and more.
  Type  help  to see all commands.

  You ▶  4
───────────────────────────────────────────────────────
  🔄  Book Renewal
───────────────────────────────────────────────────────
  Enter your Student ID (e.g. STU001): STU002

  ✅ Renewal successful for Paul Kale!

  📕 Calculus Volume 1
     Extended by 14 days  →  new due: 16 day(s) from today

  ℹ  Note: Only one renewal is allowed per book per semester.

  You ▶  1
───────────────────────────────────────────────────────
  📋  Overdue Fine Calculator
───────────────────────────────────────────────────────
  Enter your Student ID (e.g. STU001): STU002

  ✅ Paul Kale, you have NO overdue books. All returns are on time!

  You ▶  3
───────────────────────────────────────────────────────
  🎓  My Librar